# search

In [1]:
import asyncio
from src.search import match_product

In [2]:

result = await match_product(
    product_name="Raid Rapid Action Wasp, Mosquito & Fly Killer Aerosol Spray 300ml",
    website="tesco",
    country="uk",
)
print(result.verdict)                          # FinalVerdict.MATCH / NO_MATCH
print(result.matched_candidate.url if result.matched_candidate else "not found")
print(result.layer_trace.to_dict())            # per-layer pass/fail/unknown
print(result.reason)                           # LLM rationale or pipeline statusdemo())

/Users/kumo/programming/competitor_product_search/.venv/lib/python3.12/site-packages/quantulum3/classifier.py:27: UserWarning: Classifier dependencies not installed. Run `uv sync --extra classifier` or `pip install quantulum3[classifier]` to install them. The classifer helps to dissambiguate units.
  warnings.warn(


FinalVerdict.MATCH
https://www.tesco.com/shop/en-GB/products/255284088
{'domain': 'pass', 'brand': 'pass', 'numeric': 'pass', 'distinguishing': 'pass'}
Candidate 0 is Raid's Rapid Action 300ml aerosol for flies/wasps/mosquitoes, matching the query's line and size, while the others are the Eucalyptus variant or Wasp & Hornet-only product. (via duckduckgo)


# scraping

In [3]:
import asyncio
from src.scraping import scrape
from pprint import pprint

In [4]:
url_1 = "https://www.tesco.com/groceries/en-GB/products/258786820"
result = await scrape(url_1)

pprint(vars(result))

{'availability_raw': 'Out of stock',
 'brand': 'RAID',
 'currency': 'GBP',
 'gtin': '05000204146837',
 'image_urls': ['https://digitalcontent.api.tesco.com/v2/media/ghs/a6aebc5d-f15d-491e-9d64-ae73f833f450/22f53d9a-4c18-42ec-904c-f263b881e9a2_1884198801.jpeg',
                'https://digitalcontent.api.tesco.com/v2/media/ghs/ee2507a6-a07e-4cdf-a522-440bd3fa1e5b/1949fa22-5cb1-4105-8c63-6f72f5249813_536632347.jpeg',
                'https://digitalcontent.api.tesco.com/v2/media/ghs/6bc8276d-3ffd-4de3-9c07-2654dc4dc046/527519ed-1d87-4155-a4f4-2e7132f611f3_1438725754.jpeg'],
 'in_stock': False,
 'list_price': None,
 'membership_price': None,
 'parser_version': 'cs_20260818_200257',
 'price': Decimal('4.50'),
 'raw': None,
 'scraped_at': datetime.datetime(2026, 9, 13, 22, 26, 11, 986721, tzinfo=datetime.timezone.utc),
 'source_type': 'html',
 'title': 'Raid Wasp, Mosquito & Fly Killer Eucalyptus Aerosol Spray 300ml',
 'url': 'https://www.tesco.com/groceries/en-GB/products/258786820',
 'var

# match

##  load the image


In [5]:
image_urls = ['https://images3.joy-sourcing.com/product/s1248x1248_jfsintlpro-000-com/t1/4294967296/5636096/61097966189988/1139760/68e54592E52b01a56/4c765c6ab87a3a67.png.webp']

In [6]:
from image_load_compression import normalize_batch, load_config

results = await normalize_batch(image_urls, load_config(), run_id="nb")


In [8]:
import hashlib, json
from datetime import datetime, timezone
from pathlib import Path

def save_results(results, output_dir="output", run_id=None):
    """按 CLI 的约定落盘：图片 + results.jsonl"""
    run_id = run_id or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    target = Path(output_dir) / run_id
    target.mkdir(parents=True, exist_ok=True)

    for r in results:
        if r.image_bytes:
            digest = hashlib.sha256(r.url.encode()).hexdigest()[:16]
            ext = {"JPEG": "jpg"}.get(r.output_format or "", (r.output_format or "bin").lower())
            (target / f"{digest}.{ext}").write_bytes(r.image_bytes)

    with (target / "results.jsonl").open("w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r.to_dict(), ensure_ascii=False) + "\n")
    return target

save_results(results, output_dir="output", run_id="nb")


PosixPath('output/nb')